# v003 · Demo: tạo fake từ video real + phân tích video

Chạy ở **phiên tương tác** (GPU, bật Internet), rồi mở link `gradio.live` in ra ở cell cuối.

| Tab | Làm gì |
|---|---|
| 1. Tạo fake | Tải video real ≤ 60 s → chọn ô ③④⑤⑥, full/partial, generator, hệ giọng → video fake + nhãn thật |
| 2. Phân tích | Tải video (hoặc dùng video vừa tạo) → sửa độ lệch tiếng-hình → điểm 4 tác vụ của từng model + timeline, vẽ chồng nhãn thật |

Cần: output train (`runs/<RUN_NAME>/`, có `best.pt` và tốt nhất là `metrics.json` để có ngưỡng). Chưa có thì để
`RUN_DIR = None`: demo chỉ chạy `speechforensics` (không cần train).
Video người thật: chỉ dùng video của mình hoặc đã xin phép; link gradio.live là công khai trong 72 giờ, tắt khi xong.

In [ ]:
from pathlib import Path
import subprocess, sys, os, time, json, shutil
SESSION_START = time.time()
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
REPO_URL = "https://github.com/linhxm/vn-av-df-forensics.git"  # Sửa thành repository của bạn.
ROOT = Path("/kaggle/working/vn-av-df-forensics")
WORKERS = Path("/kaggle/temp/vn-av-df-workers")  # Môi trường worker: ngoài Output 20 GB.
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
# Weight và repo upstream (vài GB) ở /kaggle/temp: Output (giới hạn 20 GB) không bao giờ chứa
# chúng, kể cả khi phiên lỗi giữa chừng (bước dọn Output cuối không chạy).
for name in ("weights", "external"):
    (Path("/kaggle/temp/vn-av-df-heavy") / name).mkdir(parents=True, exist_ok=True)
    if not (ROOT / name).exists():
        (ROOT / name).symlink_to(Path("/kaggle/temp/vn-av-df-heavy") / name, target_is_directory=True)
sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[data,generation,training,demo]"],
               check=True)
import settings
settings.bootstrap()

In [ ]:
RUN_DIR = None        # Path("/kaggle/input/<output train>/vn-av-df-forensics/runs/train_part1_v003")
MODELS = None         # None = mọi model có trong RUN_DIR; hoặc ["trihead_v2", "avh_align_plus", "lipinc_v2"]
GENERATORS = ["wav2lip_gan", "latentsync_1_5"]  # generator cài sẵn cho tab "Tạo fake"
VOICES = ["f5_vi"]                               # hệ giọng cài sẵn
import torch
settings.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cfg = settings.config()

In [ ]:
def ensure_env(name):
    """Cài (hoặc dùng lại) môi trường worker và trả python của nó."""
    try:
        out = subprocess.run([sys.executable, str(ROOT / "environments/worker_envs.py"), name,
                              "--root", str(ROOT), "--env-root", str(WORKERS)],
                             check=True, capture_output=True, text=True)
        print(out.stdout[-2000:])
    except subprocess.CalledProcessError as exc:
        log = WORKERS / ".kaggle-tools" / name / "setup.log"
        if log.exists():
            print("\n".join(log.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]))
        raise RuntimeError(f"Cài môi trường {name} lỗi; xem log ở trên") from exc
    return str(WORKERS / f".venv-{name}/bin/python")

# Model nhận dạng mặt / SyncNet của data pipeline (đo + sửa độ lệch tiếng-hình)
subprocess.run([sys.executable, str(ROOT / "data_pipeline/steps/00_setup.py")], check=True)
from vn_av_df.cli import setup
print(setup(cfg, GENERATORS, encoders=True))  # source + weight generator, AV-HuBERT Large, XLS-R
subprocess.run([sys.executable, str(ROOT / "environments/bootstrap_worker.py"), "avhubert",
                "--root", str(ROOT), "--env-root", str(WORKERS)], check=True)
cfg["encoders"]["avhubert"]["python"] = str(WORKERS / ".venv-avhubert/bin/python")
from vn_av_df.generation.lipsync import ENVS as GEN_ENVS
from vn_av_df.generation.voice.engines import envs_for
if "musetalk_1_5" in GENERATORS:
    subprocess.run([sys.executable, str(ROOT / "environments/bootstrap_worker.py"), "musetalk",
                    "--root", str(ROOT), "--env-root", str(WORKERS)], check=True)
    cfg["musetalk"]["python"] = str(WORKERS / ".venv-musetalk/bin/python")
for name in GENERATORS:
    if name in GEN_ENVS:
        cfg["envs"][GEN_ENVS[name]] = ensure_env(GEN_ENVS[name])
for env in envs_for(set(VOICES)):
    cfg["envs"][env] = ensure_env(env)
print(cfg["envs"])

In [ ]:
from vn_av_df.demo.app import build_app
from vn_av_df.demo.detect import Detector, training_free_run
run_dir = RUN_DIR or training_free_run(Path("/kaggle/temp/demo-run"))
detector = Detector(cfg, run_dir, MODELS)
print("Model:", list(detector.models))
app = build_app(cfg, detector, GENERATORS, VOICES, work=Path("/kaggle/temp/demo"))
app.launch(share=True, debug=True)